In [1]:
import os
os.chdir('/home/elious/research_projects/mdpi_sensors_2026')

# Notebook 10: SHAP for Best Model + Permutation Importance

Best model per dataset (from master_results_all.csv):
- UGRansome2024 -> XGBoost
- CICIoT2023 -> RandomForest (SHAP reused from NB05)

Three importance views are compared per dataset:
- SHAP: mean absolute SHAP value (model-agnostic attribution)
- Tree built-in importance (column `tree_importance`, with `tree_importance_type` recording the method):
  - UGRansome2024 (XGBoost): gain-based importance (tree_importance_type = `xgboost_gain`)
  - CICIoT2023 (RandomForest): Gini-based mean decrease in impurity (tree_importance_type = `rf_gini`)
- Permutation importance: n_repeats=10, scoring='f1_macro', random_state=42
  (f1_macro matches the primary model-selection metric; previously this used positive-class f1)

Spearman rank correlations are computed over ALL features and written to
`results/spearman_correlations.csv`. Canonical per-dataset tables are written to
`results/shap_vs_gini_permutation_{ugr,cic}.csv`. The `threeway_importance_{ugr,cic}.csv`
files are retained as non-canonical aliases for backward compatibility.

In [2]:
import numpy as np
import pandas as pd
import joblib
import shap
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.inspection import permutation_importance
import warnings
warnings.filterwarnings('ignore')
print('Libraries loaded.')
import matplotlib
matplotlib.rcParams['savefig.dpi'] = 600
matplotlib.rcParams['figure.dpi'] = 150


Libraries loaded.


In [3]:
DPI = 300

def extract_class1(sv):
    if isinstance(sv, list):
        return sv[1]
    if sv.ndim == 3:
        return sv[:, :, 1]
    return sv

def save_fig(name):
    for ext in ('png', 'pdf'):
        plt.savefig(f'figures/{name}.{ext}', dpi=DPI, bbox_inches='tight')
    plt.close()
    print(f'  saved figures/{name}')

## UGRansome , XGBoost SHAP

In [4]:
ugr_test = pd.read_csv('data/processed/ugr_test.csv')
X_te_ugr = ugr_test.drop(columns=['Prediction'])
y_te_ugr = ugr_test['Prediction']
feat_ugr = X_te_ugr.columns.tolist()

xgb_ugr = joblib.load('results/models/ugr_XGBoost.joblib')
print('UGR XGBoost loaded. Test shape:', X_te_ugr.shape)

print('Computing UGR XGBoost SHAP ...')
ex_ugr = shap.TreeExplainer(xgb_ugr)
sv_xgb_ugr = ex_ugr.shap_values(X_te_ugr)
sv_xgb_ugr = extract_class1(sv_xgb_ugr)
print('SHAP shape:', sv_xgb_ugr.shape)

shap_imp_ugr = pd.Series(np.abs(sv_xgb_ugr).mean(axis=0), index=feat_ugr)
np.save('results/shap_values_ugr_xgb.npy', sv_xgb_ugr)

UGR XGBoost loaded. Test shape: (17972, 49)
Computing UGR XGBoost SHAP ...


SHAP shape: (17972, 49)


In [5]:
# Permutation importance , UGR XGBoost
print('Computing UGR permutation importance ...')
ugr_train = pd.read_csv('data/processed/ugr_train.csv')
X_tr_ugr  = ugr_train.drop(columns=['Prediction'])
y_tr_ugr  = ugr_train['Prediction']

perm_ugr = permutation_importance(
    xgb_ugr, X_te_ugr, y_te_ugr,
    n_repeats=10, scoring='f1_macro', random_state=42
)
perm_imp_ugr = pd.Series(perm_ugr.importances_mean, index=feat_ugr)
print('UGR permutation done.')

Computing UGR permutation importance ...


UGR permutation done.


In [6]:
# Tree built-in importance from XGBoost (gain-based for the UGRansome best model)
from scipy.stats import spearmanr

CANON_COLS = ['feature', 'shap_importance', 'shap_rank',
              'tree_importance', 'tree_importance_type', 'tree_rank',
              'perm_importance', 'perm_rank']

TREE_TYPE_UGR = 'xgboost_gain'
tree_imp_ugr = pd.Series(xgb_ugr.feature_importances_, index=feat_ugr)

df3_ugr = pd.DataFrame({
    'feature': feat_ugr,
    'shap_importance': shap_imp_ugr.values,
    'tree_importance': tree_imp_ugr.values,
    'perm_importance': perm_imp_ugr.values,
})
df3_ugr['shap_rank'] = df3_ugr['shap_importance'].rank(ascending=False).astype(int)
df3_ugr['tree_rank'] = df3_ugr['tree_importance'].rank(ascending=False).astype(int)
df3_ugr['perm_rank'] = df3_ugr['perm_importance'].rank(ascending=False).astype(int)
df3_ugr['tree_importance_type'] = TREE_TYPE_UGR
df3_ugr = df3_ugr.sort_values('shap_importance', ascending=False).reset_index(drop=True)
df3_ugr = df3_ugr[CANON_COLS]

# Spearman rank correlations over ALL features
rho_st, p_st = spearmanr(df3_ugr['shap_rank'], df3_ugr['tree_rank'])
rho_sp, p_sp = spearmanr(df3_ugr['shap_rank'], df3_ugr['perm_rank'])
rho_tp, p_tp = spearmanr(df3_ugr['tree_rank'], df3_ugr['perm_rank'])
spearman_rows_ugr = [
    {'dataset': 'UGRansome2024', 'comparison': 'SHAP vs Tree',        'tree_importance_type': TREE_TYPE_UGR, 'rho': rho_st, 'p_value': p_st},
    {'dataset': 'UGRansome2024', 'comparison': 'SHAP vs Permutation', 'tree_importance_type': TREE_TYPE_UGR, 'rho': rho_sp, 'p_value': p_sp},
    {'dataset': 'UGRansome2024', 'comparison': 'Tree vs Permutation', 'tree_importance_type': TREE_TYPE_UGR, 'rho': rho_tp, 'p_value': p_tp},
]
print(f"UGR Spearman (all features) - SHAP-Tree={rho_st:.4f}, SHAP-Perm={rho_sp:.4f}, Tree-Perm={rho_tp:.4f}")

df3_ugr.to_csv('results/shap_vs_gini_permutation_ugr.csv', index=False)
df3_ugr.to_csv('results/threeway_importance_ugr.csv', index=False)  # non-canonical alias
print('Saved results/shap_vs_gini_permutation_ugr.csv (canonical) + threeway_importance_ugr.csv (alias)')
df3_ugr.head(20)

UGR Spearman (all features) - SHAP-Tree=0.7733, SHAP-Perm=0.7787, Tree-Perm=0.7892
Saved results/shap_vs_gini_permutation_ugr.csv (canonical) + threeway_importance_ugr.csv (alias)


,feature,shap_importance,shap_rank,tree_importance,tree_importance_type,tree_rank,perm_importance,perm_rank
0,USD,2.460833,1,0.026235,xgboost_gain,5,0.075996,3
1,Clusters_2,2.440739,2,0.437801,xgboost_gain,1,0.153240,1
2,Netflow_Bytes,1.693000,3,0.010004,xgboost_gain,12,0.068646,4
3,Flag_APS,1.559082,4,0.242510,xgboost_gain,2,0.147051,2
4,Family_Globe,0.551293,5,0.025251,xgboost_gain,6,0.017241,5
5,Clusters_1,0.431929,6,0.005030,xgboost_gain,16,0.000309,22
6,Flag_APSF,0.418814,7,0.020192,xgboost_gain,7,0.017191,6
7,Time,0.384718,8,0.001320,xgboost_gain,32,0.005098,10
8,Port,0.374214,9,0.001471,xgboost_gain,29,0.000073,27
9,Family_SamSam,0.369119,10,0.005703,xgboost_gain,14,0.008132,9


## CICIoT , RF (best model) , reuse NB05 SHAP, add permutation

In [7]:
cic_test = pd.read_csv('data/processed/cic_test.csv')
X_te_cic = cic_test.drop(columns=['label', 'label_binary'])
y_te_cic = cic_test['label_binary']
feat_cic = X_te_cic.columns.tolist()

rf_cic = joblib.load('results/models/cic_RandomForest.joblib')

# Load SHAP from NB05
sv_cic = np.load('results/shap_values_cic_rf.npy')
print('CIC RF SHAP shape (from NB05):', sv_cic.shape)
shap_imp_cic = pd.Series(np.abs(sv_cic).mean(axis=0), index=feat_cic)

CIC RF SHAP shape (from NB05): (39995, 46)


In [8]:
print('Computing CIC permutation importance ...')
perm_cic = permutation_importance(
    rf_cic, X_te_cic, y_te_cic,
    n_repeats=10, scoring='f1_macro', random_state=42
)
perm_imp_cic = pd.Series(perm_cic.importances_mean, index=feat_cic)
print('CIC permutation done.')

Computing CIC permutation importance ...


CIC permutation done.


In [9]:
# Tree built-in importance from RandomForest (Gini MDI for the CICIoT best model)
TREE_TYPE_CIC = 'rf_gini'
tree_imp_cic = pd.Series(rf_cic.feature_importances_, index=feat_cic)

df3_cic = pd.DataFrame({
    'feature': feat_cic,
    'shap_importance': shap_imp_cic.values,
    'tree_importance': tree_imp_cic.values,
    'perm_importance': perm_imp_cic.values,
})
df3_cic['shap_rank'] = df3_cic['shap_importance'].rank(ascending=False).astype(int)
df3_cic['tree_rank'] = df3_cic['tree_importance'].rank(ascending=False).astype(int)
df3_cic['perm_rank'] = df3_cic['perm_importance'].rank(ascending=False).astype(int)
df3_cic['tree_importance_type'] = TREE_TYPE_CIC
df3_cic = df3_cic.sort_values('shap_importance', ascending=False).reset_index(drop=True)
df3_cic = df3_cic[CANON_COLS]

rho_st, p_st = spearmanr(df3_cic['shap_rank'], df3_cic['tree_rank'])
rho_sp, p_sp = spearmanr(df3_cic['shap_rank'], df3_cic['perm_rank'])
rho_tp, p_tp = spearmanr(df3_cic['tree_rank'], df3_cic['perm_rank'])
spearman_rows_cic = [
    {'dataset': 'CICIoT2023', 'comparison': 'SHAP vs Tree',        'tree_importance_type': TREE_TYPE_CIC, 'rho': rho_st, 'p_value': p_st},
    {'dataset': 'CICIoT2023', 'comparison': 'SHAP vs Permutation', 'tree_importance_type': TREE_TYPE_CIC, 'rho': rho_sp, 'p_value': p_sp},
    {'dataset': 'CICIoT2023', 'comparison': 'Tree vs Permutation', 'tree_importance_type': TREE_TYPE_CIC, 'rho': rho_tp, 'p_value': p_tp},
]
print(f"CIC Spearman (all features) - SHAP-Tree={rho_st:.4f}, SHAP-Perm={rho_sp:.4f}, Tree-Perm={rho_tp:.4f}")

df3_cic.to_csv('results/shap_vs_gini_permutation_cic.csv', index=False)
df3_cic.to_csv('results/threeway_importance_cic.csv', index=False)  # non-canonical alias
print('Saved results/shap_vs_gini_permutation_cic.csv (canonical) + threeway_importance_cic.csv (alias)')
df3_cic.head(20)

CIC Spearman (all features) - SHAP-Tree=0.9330, SHAP-Perm=0.9102, Tree-Perm=0.8068
Saved results/shap_vs_gini_permutation_cic.csv (canonical) + threeway_importance_cic.csv (alias)


,feature,shap_importance,shap_rank,tree_importance,tree_importance_type,tree_rank,perm_importance,perm_rank
0,rst_count,0.065082,1,0.203433,rf_gini,1,0.385445,1
1,IAT,0.061223,2,0.023123,rf_gini,14,0.336865,2
2,urg_count,0.052081,3,0.117122,rf_gini,2,0.191747,3
3,Header_Length,0.034331,4,0.042345,rf_gini,10,0.104122,4
4,Magnitue,0.029426,5,0.064901,rf_gini,4,0.036531,8
5,flow_duration,0.026853,6,0.042849,rf_gini,9,0.082257,5
6,Weight,0.024571,7,0.021687,rf_gini,16,0.016874,12
7,AVG,0.022845,8,0.050815,rf_gini,8,0.019203,11
8,Max,0.022478,9,0.055832,rf_gini,6,0.027449,10
9,Variance,0.021079,10,0.071225,rf_gini,3,0.016378,13


In [10]:
# Canonical Spearman rank-correlation table (all features, both datasets)
spearman_df = pd.DataFrame(spearman_rows_ugr + spearman_rows_cic)
spearman_df = spearman_df[['dataset', 'comparison', 'tree_importance_type', 'rho', 'p_value']]
spearman_df.to_csv('results/spearman_correlations.csv', index=False)
print('Saved results/spearman_correlations.csv')
print(spearman_df.to_string(index=False))

Saved results/spearman_correlations.csv
      dataset          comparison tree_importance_type      rho      p_value
UGRansome2024        SHAP vs Tree         xgboost_gain 0.773340 7.368071e-11
UGRansome2024 SHAP vs Permutation         xgboost_gain 0.778732 4.464954e-11
UGRansome2024 Tree vs Permutation         xgboost_gain 0.789240 1.613894e-11
   CICIoT2023        SHAP vs Tree              rf_gini 0.933040 3.729362e-21
   CICIoT2023 SHAP vs Permutation              rf_gini 0.910208 1.869334e-18
   CICIoT2023 Tree vs Permutation              rf_gini 0.806802 1.288654e-11


In [11]:
# SHAP interaction values , 1000-sample subset, seed 42
rng_int = np.random.default_rng(42)
idx_sub = rng_int.choice(len(X_te_ugr), size=1000, replace=False)
X_sub_ugr = X_te_ugr.iloc[idx_sub]

print('Computing UGR XGBoost SHAP interaction values (1000 samples) ...')
sv_interact_ugr = ex_ugr.shap_interaction_values(X_sub_ugr)
if isinstance(sv_interact_ugr, list):
    sv_interact_ugr = sv_interact_ugr[1]
elif sv_interact_ugr.ndim == 4:
    sv_interact_ugr = sv_interact_ugr[:, :, :, 1]
np.save('results/shap_interaction_ugr_xgb.npy', sv_interact_ugr)
print('Saved results/shap_interaction_ugr_xgb.npy, shape:', sv_interact_ugr.shape)

Computing UGR XGBoost SHAP interaction values (1000 samples) ...


Saved results/shap_interaction_ugr_xgb.npy, shape: (1000, 49, 49)


In [12]:
print('Notebook 10 complete.')

Notebook 10 complete.
